# 07 - Advanced Models

### Week 8:
- Train XGBoost and LightGBM regressors
- Tune hyperparameters (learning rate, depth, estimators, regularization)
- Compare against the Random Forest baseline from 04
- Document which model wins and why

### Set up and load data:

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_percentage_error
import xgboost as xgb
import lightgbm as lgb

In [2]:
train_df = pd.read_csv("data/train_final.csv")
test_df = pd.read_csv("data/test_final.csv")

drop_from_features = ['ClosePrice', 'ClosePrice_log', 'CloseDate', 'CloseYearMonth']

X_train =  train_df.drop(columns = [c for c in drop_from_features if c in  train_df.columns])
X_test = test_df.drop(columns = [ c for c in drop_from_features if c in test_df.columns])
X_test = test_df.reindex(columns = X_train.columns, fill_value=0)

y_train = train_df['ClosePrice_log']
y_test = test_df['ClosePrice_log']

actual_price = np.exp(y_test)

In [3]:
def evaluate(name, pred_log):
    pred_price = np.exp(pred_log)
    r2 = r2_score(y_test, pred_log)
    mape = mean_absolute_percentage_error(actual_price, pred_price)
    mdape = np.median(np.abs((actual_price - pred_price)/actual_price))
    print(f"{name} - R^2: {r2:.4f}, MAPE: {mape:.4f}, MdAPE: {mdape:.4f}")
    return r2, mape, mdape

print("Train: ", X_train.shape, '| Test: ', X_test.shape)


Train:  (230240, 3997) | Test:  (12851, 3997)


### XGBoost baseline (default-ish params):

In [4]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)
xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_r2, xgb_mape, xgb_mdape = evaluate('XGBoost', xgb_pred)

XGBoost - R^2: 0.9244, MAPE: 0.1347, MdAPE: 0.0915


### LightGBM:

In [6]:
import re
X_train_lgb = X_train.copy()
X_test_lgb = X_test.copy()
clean_cols = [re.sub(r'[^A-Za-z0-9_]+', '_', str(c)) for c in X_train.columns]

# dedupe in case cleaning collides names
seen = {}
final_cols = []
for c in clean_cols:
    if c in seen:
        seen[c] += 1
        final_cols.append(f"{c}_{seen[c]}")
    else:
        seen[c] = 0
        final_cols.append(c)

X_train_lgb.columns = final_cols
X_test_lgb.columns = final_cols
print('Renamed', sum(a != b for a, b in zip(X_train.columns, final_cols)), 'columns')


Renamed 2530 columns


In [8]:
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(X_train_lgb, y_train)
lgb_pred = lgb_model.predict(X_test_lgb)
lgb_r2, lgb_mape, lgb_mdape = evaluate('LightGBM', lgb_pred)

LightGBM - R^2: 0.9294, MAPE: 0.1301, MdAPE: 0.0894


### Validation split:

In [10]:
# Carve a validation set from train — most recent month before the test month
train_df['CloseYearMonth'] = pd.to_datetime(train_df['CloseDate']).dt.to_period('M')
val_month = train_df['CloseYearMonth'].max()
print('Validation month:', val_month)

val_mask = train_df['CloseYearMonth'] == val_month
X_tr = X_train[~val_mask.values]
X_val = X_train[val_mask.values]
y_tr = y_train[~val_mask.values]
y_val = y_train[val_mask.values]

print('Sub-train:', X_tr.shape, '| Val:', X_val.shape)

Validation month: 2026-05
Sub-train: (218225, 3997) | Val: (12015, 3997)


In [12]:
import itertools, re

# clean column names for lgb
def clean(df):
    d = df.copy()
    d.columns = final_cols
    return d

X_tr_lgb, X_val_lgb = clean(X_tr), clean(X_val)
actual_val = np.exp(y_val)

param_grid = {
    'num_leaves': [31, 63, 127],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [500, 1000],
    'min_child_samples': [20, 50]
}

results = []
keys = list(param_grid)
for combo in itertools.product(*param_grid.values()):
    params = dict(zip(keys, combo))
    m = lgb.LGBMRegressor(
        **params, subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1
    )
    m.fit(X_tr_lgb, y_tr)
    p = m.predict(X_val_lgb)
    pp = np.exp(p)
    results.append({
        **params,
        'r2': r2_score(y_val, p),
        'mape': mean_absolute_percentage_error(actual_val, pp),
        'mdape': np.median(np.abs((actual_val - pp) / actual_val))
    })
    print(params, f"-> R²: {results[-1]['r2']:.4f}, MdAPE: {results[-1]['mdape']:.4f}")

res_df = pd.DataFrame(results).sort_values('mdape')
print()
print(res_df.head(10))

{'num_leaves': 31, 'learning_rate': 0.05, 'n_estimators': 500, 'min_child_samples': 20} -> R²: 0.9144, MdAPE: 0.0934
{'num_leaves': 31, 'learning_rate': 0.05, 'n_estimators': 500, 'min_child_samples': 50} -> R²: 0.9140, MdAPE: 0.0935
{'num_leaves': 31, 'learning_rate': 0.05, 'n_estimators': 1000, 'min_child_samples': 20} -> R²: 0.9231, MdAPE: 0.0866
{'num_leaves': 31, 'learning_rate': 0.05, 'n_estimators': 1000, 'min_child_samples': 50} -> R²: 0.9230, MdAPE: 0.0869
{'num_leaves': 31, 'learning_rate': 0.1, 'n_estimators': 500, 'min_child_samples': 20} -> R²: 0.9224, MdAPE: 0.0894
{'num_leaves': 31, 'learning_rate': 0.1, 'n_estimators': 500, 'min_child_samples': 50} -> R²: 0.9215, MdAPE: 0.0876
{'num_leaves': 31, 'learning_rate': 0.1, 'n_estimators': 1000, 'min_child_samples': 20} -> R²: 0.9280, MdAPE: 0.0855
{'num_leaves': 31, 'learning_rate': 0.1, 'n_estimators': 1000, 'min_child_samples': 50} -> R²: 0.9276, MdAPE: 0.0830
{'num_leaves': 63, 'learning_rate': 0.05, 'n_estimators': 500, '

In [13]:
best_params = {
    'num_leaves': 127,
    'learning_rate': 0.05,
    'n_estimators': 1000,
    'min_child_samples': 50
}

lgb_tuned = lgb.LGBMRegressor(
    **best_params, subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1
)
lgb_tuned.fit(X_train_lgb, y_train)

tuned_pred = lgb_tuned.predict(X_test_lgb)
tuned_r2, tuned_mape, tuned_mdape = evaluate('LightGBM (tuned)', tuned_pred)

LightGBM (tuned) - R^2: 0.9400, MAPE: 0.1176, MdAPE: 0.0776


In [14]:
final_comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest',
              'XGBoost', 'LightGBM', 'LightGBM (tuned)'],
    'R2': [0.7888, 0.8016, 0.8979, xgb_r2, lgb_r2, tuned_r2],
    'MAPE': [0.2397, 0.2216, 0.1525, xgb_mape, lgb_mape, tuned_mape],
    'MdAPE': [0.1692, 0.1510, 0.1023, xgb_mdape, lgb_mdape, tuned_mdape]
})
print(final_comparison.round(4))
print()

# error by price band
tuned_price = np.exp(tuned_pred)
err = pd.DataFrame({
    'actual': np.asarray(actual_price),
    'ape': np.abs((np.asarray(actual_price) - tuned_price) / np.asarray(actual_price))
})
err['band'] = pd.cut(err['actual'],
    bins=[0, 300_000, 600_000, 1_000_000, 2_000_000, 5_000_000, np.inf],
    labels=['<300K', '300-600K', '600K-1M', '1-2M', '2-5M', '5M+'])

print(err.groupby('band', observed=True).agg(
    n=('ape','size'), mape=('ape','mean'), mdape=('ape','median')
).round(4))

               Model      R2    MAPE   MdAPE
0  Linear Regression  0.7888  0.2397  0.1692
1      Decision Tree  0.8016  0.2216  0.1510
2      Random Forest  0.8979  0.1525  0.1023
3            XGBoost  0.9244  0.1347  0.0914
4           LightGBM  0.9294  0.1301  0.0894
5   LightGBM (tuned)  0.9400  0.1176  0.0776

             n    mape   mdape
band                          
<300K      301  0.3917  0.2056
300-600K  2593  0.1030  0.0643
600K-1M   4248  0.0925  0.0624
1-2M      3917  0.1155  0.0874
2-5M      1540  0.1411  0.1113
5M+        252  0.2519  0.2100


# 07 - Advanced Models — Summary

**What this notebook does:**
1. Trains XGBoost and LightGBM on the same feature set as 04
2. Tunes LightGBM via grid search on a temporal validation split (2026-05, carved from train)
3. Retrains the best configuration on full train and evaluates on the held-out test month (2026-06)

**Results:**

| Model | R² | MAPE | MdAPE |
|---|---|---|---|
| Linear Regression | 0.7888 | 0.2397 | 0.1692 |
| Decision Tree | 0.8016 | 0.2216 | 0.1510 |
| Random Forest | 0.8979 | 0.1525 | 0.1023 |
| XGBoost | 0.9244 | 0.1347 | 0.0914 |
| LightGBM | 0.9294 | 0.1301 | 0.0894 |
| **LightGBM (tuned)** | **0.9400** | **0.1176** | **0.0776** |

Best parameters: `num_leaves=127, learning_rate=0.05, n_estimators=1000, min_child_samples=50`

**Takeaway:** Gradient boosting beats Random Forest substantially, and tuning adds a further ~1 point of R². Median error drops from 16.9% at the Linear Regression baseline to 7.8%. Boosting fits residuals sequentially rather than averaging independent trees, which captures the interaction effects that drive price.

**Tuning notes:**
- `n_estimators=1000` beat 500 in every configuration
- `num_leaves` improved monotonically across 31 → 63 → 127; 127 was the grid ceiling and performance was still rising, so the optimum may lie beyond the range tested
- `min_child_samples` had negligible effect
- Tuning used a validation month carved from train, not the test month, so the reported test metrics remain a clean held-out estimate

**Error by price band (tuned LightGBM):**

| Band | n | MAPE | MdAPE |
|---|---|---|---|
| <300K | 301 | 0.3917 | 0.2056 |
| 300-600K | 2593 | 0.1030 | 0.0643 |
| 600K-1M | 4248 | 0.0925 | 0.0624 |
| 1-2M | 3917 | 0.1155 | 0.0874 |
| 2-5M | 1540 | 0.1411 | 0.1113 |
| 5M+ | 252 | 0.2519 | 0.2100 |

Every band improved over Random Forest, but the distribution of error is unchanged: `<300K` and `5M+` remain 3-4x worse than mid-range. These are data problems rather than model problems — likely residual below-market transfers at the bottom and sparse training data at the top.

**Known limitations:**
- LightGBM required sanitized column names (special characters in one-hot dummy names are unsupported)
- Grid search covered 24 combinations on a single validation month; a wider grid or cross-validated time-series splits would give a more robust parameter estimate